# distillPDF — OCR for scanned PDFs

`distillpdf` turns a PDF into clean, LLM-ready **HTML/Markdown**. This notebook shows the
**OCR** feature for *image-only / scanned* pages:

1. **Detect** which pages need OCR (image-only, sparse text, or an untrusted scanner text layer).
2. **Run** a vision model (granite-docling) on those pages → **DocTags**.
3. **Render** the DocTags through distillPDF's normal HTML rules (headings, tables, figures).
4. **Emit** either OCR-augmented HTML **or** a clean, searchable PDF (rasters dropped, real
   selectable text + cropped figures).

The whole core — detection, DocTags parsing, HTML/PDF rendering — is **pure Rust**. Only the
model inference is optional and pluggable (the `[ocr]` extra).

> Round-trip invariant: `to_html(in.pdf)` ≈ `to_html(to_pdf(in.pdf))` — the searchable PDF
> preserves the document's content and structure.

## Install

The base package is a small, pure-Rust wheel with no Python runtime deps:

```bash
pip install distillpdf
```

OCR adds a model runtime (downloaded on first use). Install the extra:

```bash
pip install 'distillpdf[ocr]'   # huggingface-hub + llama-cpp-python
```

If you run OCR on a base install you get a clear, actionable error telling you exactly this.

In [ ]:
# !pip install 'distillpdf[ocr]'
import distillpdf
print('distillpdf', distillpdf.__version__)
print('OCR backends:', distillpdf.available_backends())

## 1. The pure-Rust core — DocTags → HTML / searchable PDF (no model needed)

OCR models in the *docling* family emit **DocTags**: positioned, typed elements
(`<section_header_…>`, `<text>`, `<otsl>` tables, `<picture>`…). distillPDF parses and renders
these with the *same* rules it uses for born-digital PDFs. You can render DocTags directly —
handy for testing or for piping output from any docling-compatible model:

In [ ]:
from distillpdf import ocr

# A small sample of what a model emits for one scanned page (heading, paragraph, table, logo).
SAMPLE_DOCTAGS = (
    '<section_header_level_1><loc_50><loc_30><loc_450><loc_55>QUARTERLY REPORT</section_header_level_1>'
    '<text><loc_50><loc_70><loc_460><loc_140>This document summarizes the results for the period. '
    'All figures are in thousands of USD unless otherwise noted.</text>'
    '<otsl><loc_50><loc_160><loc_460><loc_250>'
    '<ched>Quarter<ched>Revenue<ched>Profit<nl>'
    '<fcel>Q1<fcel>1200<fcel>180<nl>'
    '<fcel>Q2<fcel>1350<fcel>210<nl></otsl>'
    '<loc_50><loc_270><loc_180><loc_300><logo>'
)

html = ocr.render_doctags(SAMPLE_DOCTAGS)
print(html)

Note how the table became a real `<table>` (with `<th>` headers) and the logo a `<figure>` —
the same semantic HTML distillPDF emits for digital PDFs, so it flows into Markdown / sections /
TOC unchanged.

We can also write a **clean, searchable PDF** straight from DocTags (real selectable text):

In [ ]:
from distillpdf._distillpdf import ocr_doctags_to_pdf

# (doctags, page_image_path_or_'', width_pts, height_pts) per page; '' = no figure cropping.
ocr_doctags_to_pdf([(SAMPLE_DOCTAGS, '', 612.0, 792.0)], 'demo_clean.pdf')

# The result is a normal PDF whose text is selectable — re-extract it to confirm:
print(distillpdf.open('demo_clean.pdf').to_html(return_string=True)[:400])

## 2. OCR a real scanned PDF (needs `distillpdf[ocr]`)

### a) See which pages need OCR

`ocr_plan()` inspects every page and reports the decision (no model required — this is pure
Rust). `reason` is one of `NotNeeded`, `NeedsOcr` (image + no/sparse text) or `DropAndOcr`
(an untrusted scanner/Tesseract text layer that should be re-OCR'd).

In [ ]:
pdf = distillpdf.open('scanned.pdf')   # <-- your scanned PDF
for p in pdf.ocr_plan():
    print(f"page {p['page']:>3}  needs_ocr={p['needs_ocr']!s:<5}  {p['reason']:<10}  "
          f"{p['width_pts']:.0f}x{p['height_pts']:.0f}pt  image={'yes' if p['image'] else 'no'}")

### b) Configure the model backend

The backend is a standardized wrapper: it handles model **download/caching** (`model_dir`) and
**authentication** (`hf_token`). The default `granite-docling` backend pulls a ~190 MB GGUF on
first use (Metal/CPU via llama-cpp-python).

In [ ]:
backend = distillpdf.get_backend(
    'granite-docling',
    model_dir=None,     # None -> default HF cache (honors HF_HOME / HF_HUB_CACHE)
    hf_token=None,      # None -> ambient HF_TOKEN / cached login (only needed for gated models)
    device='auto',      # 'auto' | 'metal' | 'cuda' | 'cpu'
)
print('backend:', backend.name, '| model:', backend.config.model_id)

### c) OCR → clean HTML, and → a searchable PDF

`to_html` runs the backend only on flagged pages and splices the result into the document
(born-digital pages keep distillPDF's normal extraction). `to_pdf` rebuilds flagged pages as
real text + cropped figures and **drops the page rasters** — typically ~30× smaller, fully
searchable. Born-digital pages are kept verbatim.

In [ ]:
# OCR-augmented HTML (string, or pass path=... to write a file)
html = distillpdf.ocr.to_html(pdf, backend)
print(html[:600])

# Clean, searchable PDF
distillpdf.ocr.to_pdf(pdf, backend, 'scanned.clean.pdf')

## 3. Advanced

**Custom backend.** Any object with an `ocr_page(image_bytes) -> doctags_str` method works —
subclass `OcrBackend` to plug in a different model, a remote service, or pre-captured output:

In [ ]:
from distillpdf.ocr import OcrBackend

class MyBackend(OcrBackend):
    name = 'my-backend'
    def ocr_page(self, image: bytes) -> str:
        # call your own model / service here and return a DocTags string
        return SAMPLE_DOCTAGS

# distillpdf.ocr.to_html(pdf, MyBackend())
# distillpdf.ocr.to_pdf(pdf, MyBackend(), 'out.pdf')

**Zero-Python-dep path.** distillPDF also ships a pure-Rust `ServerOcrEngine` that talks to a
llama.cpp `llama-server` you run yourself — no `[ocr]` extra needed.

**Tables.** DocTags tables (OTSL) are already parsed and rendered (see §1). granite-docling
emits them on cleaner inputs; a future table-specialized backend (e.g. docling's TableFormer,
which also outputs OTSL) plugs into the same registry with no core changes.

---
See the [README](../README.md) for the full API.